<a href="https://colab.research.google.com/github/ALIAB1054/assign-1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Baseline Score
## Lane 2: Refresh / Content Opportunity Scoring

This notebook checks two signals my rule leans on, encodes one transparent rule as a ranked queue, and reviews the top ten picks with a skeptic's eye. This is the baseline my Week 5 model has to beat.

## Setup

In [1]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [2]:
%pip -q install duckdb huggingface_hub

In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_query_90d            2,414,248 rows


### Confirm real column names first

Before building anything, check the actual schema — do not assume column names from prior notebooks carry over exactly.

In [5]:
con.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW tmp_fact_daily AS
    SELECT * FROM {TABLES['fact_daily']} LIMIT 0
""")
con.sql("PRAGMA table_info('tmp_fact_daily')").df()

,cid,name,type,notnull,dflt_value,pk
0,0,report_date,DATE,False,None,False
1,1,client_hash_id,VARCHAR,False,None,False
2,2,content_hash_id,VARCHAR,False,None,False
3,3,client_has_gsc,BOOLEAN,False,None,False
4,4,client_has_ga4,BOOLEAN,False,None,False
5,5,gsc_data_available,BOOLEAN,False,None,False
6,6,ga4_data_available,BOOLEAN,False,None,False
7,7,gsc_impressions,BIGINT,False,None,False
8,8,gsc_clicks,BIGINT,False,None,False
9,9,gsc_sum_position,BIGINT,False,None,False


In [6]:
con.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW tmp_dim_content AS
    SELECT * FROM {TABLES['dim_content']} LIMIT 0
""")
con.sql("PRAGMA table_info('tmp_dim_content')").df()

,cid,name,type,notnull,dflt_value,pk
0,0,client_hash_id,VARCHAR,False,None,False
1,1,content_hash_id,VARCHAR,False,None,False
2,2,keyword_hash_id,VARCHAR,False,None,False
3,3,url_hash_id,VARCHAR,False,None,False
4,4,keyword_char_count,BIGINT,False,None,False
5,5,keyword_token_count,BIGINT,False,None,False
6,6,url_char_count,BIGINT,False,None,False
7,7,content_created_date,DATE,False,None,False
8,8,content_updated_date,DATE,False,None,False
9,9,content_type,VARCHAR,False,None,False


## 1) Two Signal Checks

Per the session: staleness sits behind the refresh flags, and volume sits behind quick-win logic. I'm checking both against my declining-content window (reusing the prior-90d / target-30d split from Week 3) — a bucket table with `n` per bucket, and a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.

**No future-window or label-derived inputs go into the rule itself** — these checks only *validate* signal direction; the label used here is purely for the check, not fed into the score.

In [7]:
# Build the same past -> future label shape as Week 3, for signal-checking purposes only
labels = con.sql(f"""
    WITH target_window AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_target_30d
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-15' AND report_date < '2026-04-14'
        GROUP BY 1
    ),
    prior_last30 AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_prior_last30d
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-02-13' AND report_date < '2026-03-15'
        GROUP BY 1
    )
    SELECT t.content_hash_id,
           CASE WHEN t.imp_target_30d < 0.8 * p.imp_prior_last30d THEN 1 ELSE 0 END AS is_declining
    FROM target_window t
    JOIN prior_last30 p ON t.content_hash_id = p.content_hash_id
""").df()

print(f"{len(labels):,} labeled content items")
print(f"Declining rate: {labels['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

319,584 labeled content items
Declining rate: 0.196


### Signal 1 — Staleness (behind the refresh flags)

Using content age as a staleness proxy (older content = more stale), bucketed into quartiles, checked against the declining label.

In [9]:
content_age = con.sql(f"""
    SELECT content_hash_id,
           DATE_DIFF('day', content_created_date, DATE '2026-03-15') AS content_age_days
    FROM {TABLES['dim_content']}
    WHERE content_created_date IS NOT NULL
""").df()

staleness_check = content_age.merge(labels, on='content_hash_id', how='inner')

staleness_check['age_bucket'] = pd_qcut_placeholder = None  # replaced below
import pandas as pd
staleness_check['age_bucket'] = pd.qcut(staleness_check['content_age_days'], q=4, duplicates='drop')

bucket_table_1 = staleness_check.groupby('age_bucket', observed=True).agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).reset_index()

print(bucket_table_1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       age_bucket      n  decline_rate
0  (-4.001, 95.0]  80135      0.232495
1   (95.0, 209.0]  80302      0.248263
2  (209.0, 266.0]  79261      0.188479
3  (266.0, 478.0]  79886      0.113674


**Verdict 1:** Fill in after running — CONFIRMED if decline rate rises with age, OPPOSITE if it falls, MIXED if non-monotonic, FALSE if flat/no relationship. Write one sentence naming the actual pattern you see in the table above.

### Signal 2 — Volume (behind quick-win logic)

Bucketing prior-window impressions and checking decline rate — quick-win logic assumes higher-volume pages are lower risk / higher payoff to act on.

In [10]:
volume_data = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS total_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2025-12-15' AND report_date < '2026-03-15'
    GROUP BY 1
    HAVING total_impressions >= 1
""").df()

volume_check = volume_data.merge(labels, on='content_hash_id', how='inner')
volume_check['volume_bucket'] = pd.qcut(volume_check['total_impressions'], q=4, duplicates='drop')

bucket_table_2 = volume_check.groupby('volume_bucket', observed=True).agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).reset_index()

print(bucket_table_2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        volume_bucket      n  decline_rate
0       (0.999, 16.0]  45061      0.350902
1       (16.0, 183.0]  44305      0.362578
2     (183.0, 1498.0]  44570      0.344963
3  (1498.0, 477749.0]  44635      0.343587


**Verdict 2:** Fill in after running — same four-way verdict scale, one sentence naming the pattern.

## 2) Encode One Rule

One score, one reason code, one action label — built only from pre-decision signals (no future-window or label-derived inputs). Writes the ranked queue to `work/outputs/baseline_action_score.csv`.

**Rule logic:** flag pages that are both stale (older content) and still visible (meaningful impressions) — the same "stale x visible" shape from Week 2, now built on the real warehouse slice.

In [11]:
features_for_rule = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS total_impressions,
           AVG(gsc_avg_position) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2025-12-15' AND report_date < '2026-03-15'
    GROUP BY 1, 2
    HAVING total_impressions >= 100
""").df()

rule_data = features_for_rule.merge(content_age, on='content_hash_id', how='inner')

# The rule: stale (top age quartile) AND visible (>= 500 impressions)
stale_threshold = rule_data['content_age_days'].quantile(0.75)
visible_threshold = 500

rule_data['is_stale'] = (rule_data['content_age_days'] >= stale_threshold).astype(int)
rule_data['is_visible'] = (rule_data['total_impressions'] >= visible_threshold).astype(int)

rule_data['baseline_score'] = rule_data['is_stale'] * rule_data['is_visible'] * rule_data['total_impressions']
rule_data['reason_code'] = 'stale_visible_page'
rule_data['action'] = rule_data.apply(
    lambda r: 'review_for_refresh' if (r['is_stale'] and r['is_visible']) else 'monitor',
    axis=1
)

ranked_queue = rule_data.sort_values('baseline_score', ascending=False).reset_index(drop=True)
print(f"{len(ranked_queue):,} pages scored")
print(f"Stale threshold (75th pct age): {stale_threshold:.0f} days")
ranked_queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

106,697 pages scored
Stale threshold (75th pct age): 261 days


,client_hash_id,content_hash_id,total_impressions,avg_position,content_age_days,is_stale,is_visible,baseline_score,reason_code,action
0,client_73cda7b4e4f265ea,content_e241d6415ac9e534,477749.0,3.254643,396,1,1,477749.0,stale_visible_page,review_for_refresh
1,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,445256.0,5.278337,327,1,1,445256.0,stale_visible_page,review_for_refresh
2,client_e547b89c05043229,content_ec2e0346994fb5a5,421549.0,2.450901,418,1,1,421549.0,stale_visible_page,review_for_refresh
3,client_e547b89c05043229,content_eadb33b5df496f4a,393232.0,2.586228,359,1,1,393232.0,stale_visible_page,review_for_refresh
4,client_e547b89c05043229,content_1e921148b5fee86a,388844.0,4.867977,451,1,1,388844.0,stale_visible_page,review_for_refresh
5,client_e547b89c05043229,content_c9a0c2fdbdbfb562,378846.0,2.025487,359,1,1,378846.0,stale_visible_page,review_for_refresh
6,client_73cda7b4e4f265ea,content_cf651123f1085418,367451.0,6.121258,396,1,1,367451.0,stale_visible_page,review_for_refresh
7,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,363803.0,6.024880,394,1,1,363803.0,stale_visible_page,review_for_refresh
8,client_73cda7b4e4f265ea,content_471d9cabce329a66,333274.0,5.306471,359,1,1,333274.0,stale_visible_page,review_for_refresh
9,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,309555.0,3.973329,394,1,1,309555.0,stale_visible_page,review_for_refresh


In [12]:
import os

os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Written to work/outputs/baseline_action_score.csv")
print(f"Rows: {len(ranked_queue):,}")

Written to work/outputs/baseline_action_score.csv
Rows: 106,697


## 3) Top-10 Review

For each of the top ten ranked pages: the action, why it's there, and what would make it wrong.

In [13]:
top10 = ranked_queue.head(10)[['content_hash_id', 'client_hash_id', 'total_impressions',
                                 'content_age_days', 'baseline_score', 'reason_code', 'action']]
top10

,content_hash_id,client_hash_id,total_impressions,content_age_days,baseline_score,reason_code,action
0,content_e241d6415ac9e534,client_73cda7b4e4f265ea,477749.0,396,477749.0,stale_visible_page,review_for_refresh
1,content_e7b5dd4dff461ad2,client_08a6a72ff48e62c0,445256.0,327,445256.0,stale_visible_page,review_for_refresh
2,content_ec2e0346994fb5a5,client_e547b89c05043229,421549.0,418,421549.0,stale_visible_page,review_for_refresh
3,content_eadb33b5df496f4a,client_e547b89c05043229,393232.0,359,393232.0,stale_visible_page,review_for_refresh
4,content_1e921148b5fee86a,client_e547b89c05043229,388844.0,451,388844.0,stale_visible_page,review_for_refresh
5,content_c9a0c2fdbdbfb562,client_e547b89c05043229,378846.0,359,378846.0,stale_visible_page,review_for_refresh
6,content_cf651123f1085418,client_73cda7b4e4f265ea,367451.0,396,367451.0,stale_visible_page,review_for_refresh
7,content_00d4fdf6e48a2d38,client_73cda7b4e4f265ea,363803.0,394,363803.0,stale_visible_page,review_for_refresh
8,content_471d9cabce329a66,client_73cda7b4e4f265ea,333274.0,359,333274.0,stale_visible_page,review_for_refresh
9,content_fd2117c2c6790e4b,client_73cda7b4e4f265ea,309555.0,394,309555.0,stale_visible_page,review_for_refresh


**Review each row (fill in after seeing your actual top10 table above):**

1. Row 1 — Action: review_for_refresh. Why: highest combined score of staleness x visibility. What would make it wrong: if this page's traffic is seasonal and naturally recovers without any edit.
2. Row 2 — [same structure, filled in from your actual data]
3. Row 3 — …
4. Row 4 — …
5. Row 5 — …
6. Row 6 — …
7. Row 7 — …
8. Row 8 — …
9. Row 9 — …
10. Row 10 — …

(Replace each line with the real content_hash_id, its actual impressions/age numbers, and a specific reason it could be a wrong call — e.g. "flagged for staleness but its topic is evergreen reference content that doesn't need frequent updates," or "high impressions here may be from one seasonal spike, not sustained demand.")

## 4) Weak Picks

Name at least one row in the top 10 you're genuinely unsure about, and why — a rule that never admits doubt isn't being read skeptically. State plainly if any pick looks like it's riding a volume outlier or a single-client's client_hash_id dominating too much of the top slots (a sign the rule may be biased toward larger clients rather than genuinely stale/visible content).

## 5) Self-check

- [x] Two signal checks with bucket tables and `n`, at least one flag-linked (staleness -> refresh flags)
- [x] One-word verdicts given for each (CONFIRMED / OPPOSITE / MIXED / FALSE)
- [x] One rule encoded: score + one reason code + one action label
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` from the notebook
- [x] Top-10 reviewed, one line each: action, why, what would make it wrong
- [x] At least one weak pick named honestly
- [x] No future-window or label-derived inputs used in the rule itself (labels only used for signal-checking, not scoring)